# 🔬 DermaScan — Backend API (Google Colab)

Ce notebook déploie votre modèle ensemble de classification de lésions cutanées via une API Flask + ngrok.

**Instructions :**
1. Exécutez les cellules dans l'ordre
2. Uploadez vos fichiers modèles quand demandé
3. Copiez l'URL ngrok dans votre app React Native

In [ ]:
# Cellule 1 — Installation des dépendances
!pip install flask flask-cors pyngrok pillow -q
print('✅ Dépendances installées')

In [ ]:
# Cellule 2 — Monter Google Drive (pour charger les modèles)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive monté')

In [ ]:
# Cellule 3 — Configuration des chemins
# ⚠️ MODIFIEZ ces chemins selon l'emplacement de vos modèles dans Drive
import os

# Option A: Modèles dans Google Drive
DRIVE_MODEL_DIR = '/content/drive/MyDrive/DermaScan/models'  # ← Modifiez ce chemin

# Option B: Upload direct (décommentez si vous préférez uploader)
# from google.colab import files
# uploaded = files.upload()  # Uploadez model1.h5, model2.h5, ensemble_info.json
# DRIVE_MODEL_DIR = '/content'

# Créer le dossier models local
os.makedirs('/content/models', exist_ok=True)

# Copier les fichiers
if os.path.exists(DRIVE_MODEL_DIR):
    for f in os.listdir(DRIVE_MODEL_DIR):
        src = os.path.join(DRIVE_MODEL_DIR, f)
        dst = os.path.join('/content/models', f)
        if not os.path.exists(dst):
            !cp "{src}" "{dst}"
            print(f'📁 Copié: {f}')
else:
    print(f'⚠️ Dossier non trouvé: {DRIVE_MODEL_DIR}')
    print('Uploadez vos modèles manuellement ou modifiez le chemin')

print('\n📂 Fichiers dans /content/models:')
for f in os.listdir('/content/models'):
    size = os.path.getsize(f'/content/models/{f}') / 1024 / 1024
    print(f'  {f} ({size:.1f} MB)')

In [ ]:
# Cellule 4 — Charger les modèles
import tensorflow as tf
import numpy as np
import json

tf.get_logger().setLevel('ERROR')

MODEL_DIR = '/content/models'
model1, model2, ensemble_info = None, None, None

# Trouver et charger les modèles
h5_files = [f for f in os.listdir(MODEL_DIR) if f.endswith(('.h5', '.keras'))]
json_files = [f for f in os.listdir(MODEL_DIR) if f.endswith('.json')]

print(f'Fichiers .h5/.keras trouvés: {h5_files}')
print(f'Fichiers .json trouvés: {json_files}')

if len(h5_files) >= 1:
    model1 = tf.keras.models.load_model(os.path.join(MODEL_DIR, h5_files[0]), compile=False)
    print(f'✅ Model 1: {h5_files[0]} — input={model1.input_shape}, output={model1.output_shape}')

if len(h5_files) >= 2:
    model2 = tf.keras.models.load_model(os.path.join(MODEL_DIR, h5_files[1]), compile=False)
    print(f'✅ Model 2: {h5_files[1]} — input={model2.input_shape}, output={model2.output_shape}')

if json_files:
    with open(os.path.join(MODEL_DIR, json_files[0]), 'r') as f:
        ensemble_info = json.load(f)
    print(f'✅ Ensemble info: accuracy={ensemble_info.get("accuracy", "N/A")}')

mode = 'ENSEMBLE' if model2 else ('SINGLE' if model1 else 'DEMO')
print(f'\n🔬 Mode: {mode}')

In [ ]:
# Cellule 5 — Lancer l'API Flask + ngrok
import io
import base64
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

IMG_SIZE = (224, 224)
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_NAMES_FR = {
    'akiec': 'Kératose actinique', 'bcc': 'Carcinome basocellulaire',
    'bkl': 'Kératose bénigne', 'df': 'Dermatofibrome',
    'mel': 'Mélanome', 'nv': 'Naevus mélanocytaire', 'vasc': 'Lésion vasculaire',
}
RISK_LEVELS = {
    'akiec': 'pre-malignant', 'bcc': 'malignant', 'bkl': 'benign',
    'df': 'benign', 'mel': 'malignant', 'nv': 'benign', 'vasc': 'benign',
}
CLASS_ICONS = {
    'akiec': '🔶', 'bcc': '🔴', 'bkl': '🟢',
    'df': '🟤', 'mel': '⚫', 'nv': '🟡', 'vasc': '🟣',
}

RECOMMENDATIONS_FR = {
    'akiec': 'Lésion pré-cancéreuse. Consultation dermatologique recommandée sous 2 semaines pour biopsie ou traitement (cryothérapie, PDT).',
    'bcc': 'Carcinome basocellulaire suspecté. Référer à un dermatologue pour exérèse chirurgicale. Faible risque métastatique.',
    'bkl': 'Lésion bénigne (kératose séborrhéique probable). Surveillance annuelle recommandée.',
    'df': 'Lésion bénigne (dermatofibrome). Aucun traitement nécessaire sauf gêne esthétique.',
    'mel': '⚠️ URGENT: Mélanome suspecté. Référer immédiatement à un dermatologue pour exérèse et analyse histopathologique.',
    'nv': 'Naevus bénin. Surveillance dermoscopique annuelle. Exérèse si modification rapide.',
    'vasc': 'Lésion vasculaire bénigne. Surveillance simple. Traitement laser si nécessaire.',
}

DESCRIPTIONS_FR = {
    'akiec': 'Lésion squameuse sur peau photoexposée, précurseur du carcinome épidermoïde.',
    'bcc': 'Tumeur cutanée la plus fréquente, croissance lente, aspect perlé caractéristique.',
    'bkl': 'Lésion pigmentée bénigne, fréquente après 40 ans, aspect verruqueux.',
    'df': 'Nodule dermique bénin, ferme à la palpation, signe du capiton positif.',
    'mel': 'Tumeur maligne des mélanocytes. Critères ABCDE : Asymétrie, Bords irréguliers, Couleur inhomogène, Diamètre >6mm, Évolution.',
    'nv': 'Prolifération bénigne de mélanocytes. Lésion pigmentée homogène et symétrique.',
    'vasc': 'Prolifération vasculaire bénigne (angiome, angiokeratome).',
}

def preprocess_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img = img.resize(IMG_SIZE)
    return np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)

def do_predict(img_bytes):
    img_array = preprocess_image(img_bytes)
    if model1 is not None and model2 is not None:
        p1 = model1.predict(img_array, verbose=0)
        p2 = model2.predict(img_array, verbose=0)
        c1, cf1 = int(np.argmax(p1)), float(np.max(p1))
        c2, cf2 = int(np.argmax(p2)), float(np.max(p2))
        fp = p1.copy()
        if ensemble_info and 'rules' in ensemble_info:
            if c1 == 1 and cf1 > 0.68: fp[0][1] = max(fp[0][1], 0.85)
            if c2 == 4 and cf2 < 0.8 and c1 == 4: fp[0][4] = max(fp[0][4], 0.80)
            if c2 == 0 and cf2 < 0.8 and c1 == 0: fp[0][0] = max(fp[0][0], 0.75)
        fp = fp / np.sum(fp)
        low_conf = cf2 < 0.9245
    elif model1 is not None:
        fp = model1.predict(img_array, verbose=0)
        low_conf = float(np.max(fp)) < 0.5
    else:
        fp = np.random.dirichlet(np.ones(7)).reshape(1, -1)
        low_conf = True
    results = []
    for i, prob in enumerate(fp[0].tolist()):
        code = CLASS_NAMES[i]
        results.append({
            'id': code, 'code': code,
            'name': CLASS_NAMES_FR[code], 'nameFr': CLASS_NAMES_FR[code],
            'confidence': round(prob, 6),
            'riskLevel': RISK_LEVELS[code],
            'icon': CLASS_ICONS[code],
            'descriptionFr': DESCRIPTIONS_FR[code],
            'recommendationFr': RECOMMENDATIONS_FR[code],
        })
    results.sort(key=lambda x: x['confidence'], reverse=True)
    return {'predictions': results, 'topPrediction': results[0], 'lowConfidence': low_conf,
            'modelUsed': 'ensemble' if model2 else ('single' if model1 else 'demo')}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'mode': 'ensemble' if model2 else ('single' if model1 else 'demo'),
                    'model1': model1 is not None, 'model2': model2 is not None})

@app.route('/predict', methods=['POST'])
def predict_route():
    try:
        data = request.json
        if not data or 'image' not in data:
            return jsonify({'error': 'Missing image field'}), 400
        img_b64 = data['image']
        if ',' in img_b64: img_b64 = img_b64.split(',')[1]
        result = do_predict(base64.b64decode(img_b64))
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/', methods=['GET'])
def home():
    return jsonify({'name': 'DermaScan API', 'version': '1.0.0'})

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f'\n{"="*60}')
print(f'🚀 DermaScan API is LIVE!')
print(f'📡 Public URL: {public_url}')
print(f'🔬 Mode: {"ENSEMBLE" if model2 else ("SINGLE" if model1 else "DEMO")}')
print(f'{"="*60}')
print(f'\n👉 Copiez cette URL dans votre app React Native:')
print(f'   API_URL = "{public_url}"')
print(f'\n📋 Endpoints:')
print(f'   GET  {public_url}/health')
print(f'   POST {public_url}/predict')

# Run Flask in background
threading.Thread(target=lambda: app.run(port=5000)).start()

In [ ]:
# Cellule 6 — Tester l'API (optionnel)
import requests

# Test health
r = requests.get(f'{public_url}/health')
print('Health:', r.json())

# Test avec une image de test
from PIL import Image
import io, base64

# Créer une image de test (carré rouge)
test_img = Image.new('RGB', (224, 224), color='brown')
buf = io.BytesIO()
test_img.save(buf, format='JPEG')
b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

r = requests.post(f'{public_url}/predict', json={'image': b64})
result = r.json()
print(f'\nTop prediction: {result["topPrediction"]["nameFr"]}')
print(f'Confidence: {result["topPrediction"]["confidence"]*100:.1f}%')
print(f'Risk: {result["topPrediction"]["riskLevel"]}')
print(f'Model used: {result["modelUsed"]}')
print(f'\nAll predictions:')
for p in result['predictions']:
    print(f'  {p["icon"]} {p["nameFr"]}: {p["confidence"]*100:.1f}% ({p["riskLevel"]})')